# Optimisation linéaire en nombres entiers : modélisation - Correction

In [ ]:
from mip import *

Les solveurs tels que CBC (utilisé par défaut par le package Python-MIP) peuvent résoudre des problèmes d'optimisation linéaire en nombres entiers en appliquant l'algorithme de séparation et évaluation (version améliorée pour aller beaucoup plus vite !). Le package Python-MIP permet de modéliser très facilement des problèmes d'optimisation linéaire en nombres entiers.

Un problème d'optimisation linéaire en nombres entiers se modélise comme un problème d'optmisation linéaire : il faut seulement indiquer au solveur que les variables sont binaires (Binary) ou entières (Integer). Cela se fait au moment de l'ajout d'une variable dans le modèle (méthode `add_var`) en spécifiant le type de la variable (paramètre `var_type`). Pour ajouter une variable binaire (resp. entière) au modèle `m`, il suffit de faire `m.add_var(var_type=BINARY)` (resp. `m.add_var(var_type=INTEGER)`). 

**Exemple :** Résolution du problème d'optimisation linéaire à variables mixtes :

$\max 15 x_0 + 4 x_1 + 7 y - 10 z$

$x_0 + 2x_1 \le z + 1$

$2 x_0 + 3 x_1 + y \le 5$

$x_0, x_1 \in \{0,1\}$ 

$ y $ entier

$z \ge 0$

In [ ]:
def exemple():
    m = Model()
    x = [m.add_var(var_type = BINARY) for j in range(2)]
    y = m.add_var(var_type = INTEGER, lb = -float("inf"))
    z = m.add_var()

    m.objective = maximize(15 * x[0] + 4 * x[1] + 7 * y - 10 * z)
    m += x[0] + 2 * x[1] <= z + 1
    m += 2 * x[0] + 3 * x[1] + y <= 5

    status = m.optimize()
    if status == OptimizationStatus.OPTIMAL:
        print(f"Optimum = {m.objective_value}")
        print(f"x0 = {x[0].x}, x1 = {x[1].x}, y = {y.x}, z = {z.x}")

exemple()

## Exercice 1 : Équipe de super-héros

Pour combattre les aliens qui envahissent la terre, il faut créer une équipe de super-héros travaillant main dans la main. Malheureusement, certains super-héros sont ennemis et ne peuvent donc pas faire équipe...  Combien de super-héros peuvent aller combattre les aliens ? Il faut trouver l'équipe la plus importante possible sans ennemis... Le sort de la terre en dépend !

Voici la liste des super-héros : 
   * Batman
   * Superman
   * Catwoman
   * Flash
   * Wonder woman
   * Black Panther
   * Captain America
   * Daredevil
   * Elektra
   * Hulk

Et voici maintenant la liste des ennemis jurés : 
   * Batman et Flash
   * Catwoman et Captain America
   * Daredevil et Elektra
   * Hulk et Batman
   * Catwoman et Wonder woman
   * Black Panther et Hulk
   * Superman et Flash
   * Superman et Elektra
   * Flash et Daredevil
   * Wonder woman et Captain America
   * Daredevil et Hulk
   * Batman et Captain America
   * Batman et Wonder woman
   * Black Panther et Wonder woman


**CORRECTION :**


On peut modéliser le problème de la manière suivante.

Données : 
* On note par $S$ l'ensemble des super-héros.
* On note par $E$ l'ensemble des paires d'ennemis jurés. Pour tout $e \in E$, on a $e=(s,s')$ avec $s$ et $s'$ deux super-héros distincts qui correspondent sont ennemis jurés.

Variables : 
* À chaque super-héros $s \in S$, on associe une variable binaire $x_s$ tel que :
  $$
  x_s = \left\{ \begin{array}{ll} 1 & \text{si le super-héros fait partie de l'équipe}\\ 0 & \text{sinon.} \end{array} \right.
  $$

Objectif : 
* On maximise le nombre de super-héros de l'équipe, c'est-à-dire le nombre de variables à un :
  $$
  \max \sum_{s \in S} x_s
  $$

Contraintes:
* Pour chaque paire d'ennemis jurés $e = (s, s')$ de $E$, on a la contrainte :
  $$
  x_s + x_{s'} \le 1.
  $$

In [ ]:
#############
#  DONNÉES  #
#############

Superheros = [
 "Batman",
 "Superman",
 "Catwoman",
 "Flash",
 "Wonder woman",
 "Black Panther",
 "Captain America",
 "Daredevil",
 "Elektra",
 "Hulk"
]



Ennemis = [
  ["Batman", "Flash"],
  ["Catwoman", "Captain America"],
  ["Daredevil", "Elektra"],
  ["Hulk", "Batman"],
  ["Catwoman", "Wonder woman"],
  ["Black Panther", "Hulk"],
  ["Superman", "Flash"],
  ["Superman", "Elektra"],
  ["Flash", "Daredevil"],
  ["Wonder woman", "Captain America"],
  ["Daredevil", "Hulk"],
  ["Batman", "Captain America"],
  ["Batman", "Wonder woman"],
  ["Black Panther", "Wonder woman"]
]

In [ ]:
##################
#   Correction   #
##################


def teamSH(Superheros, Ennemis):

    m = Model()
    m.verbose = 0
    
    # On crée un dictionnaire dont les clés sont les superhéros et les valeurs les variables associées telles que
    # x[h] = 1 si le superhéros est dans l'équipe, 0 sinon
    x = dict(zip(Superheros,  [m.add_var(var_type=BINARY) for h in Superheros]))

    m.objective = maximize(xsum(x[h] for h in Superheros))

    for (s1,s2) in Ennemis:
        # Les deux super-héros ennemis ne peuvent pas être ensemble
        m += x[s1] + x[s2] <= 1

    status = m.optimize()
    if status == OptimizationStatus.OPTIMAL:
        print(f"Formation d'une équipe de {round(m.objective_value)} superhéros !")
        for h in Superheros:
            if round(x[h].x) == 1:
                print(f"\t- {h}")


teamSH(Superheros, Ennemis)

## Exercice 2 : Fournisseur d'électricité


Une entreprise d'électricité doit fournir de l’électricité pour les villes de Villetaneuse, Épinay et Saint-Denis. La puissance nécessaire d'électricité pour chaque ville est respectivement de 3000, 6000 et 5000 GW. Pour produire cette électricité, l’entreprise peut construire cinq centrales électriques C1,...,C5. Le coût estimé de production et d’acheminement de l’électricité en fonction des villes et des centrales (en &euro; par GW) est récapitulée dans le tableau suivant.



|  &nbsp;    | Villetaneuse | Épinay | Saint-Denis |
| ------- | -----        | ---    | ------------|
| C1      | 80           | 100    | 120         |
| C2      | 20           | 30     | 40          |
| C3      | 40           | 60     | 80          |
| C4      | 70           | 90     | 60          |
| C5      | 30           | 30     | 50          |


La construction de chaque centrale entraîne un coût fixe. De plus, chaque usine aura une puissance limitée. Les coûts de construction et les capacités sont données dans le tableau suivant.


|  &nbsp;       | Coût de construction | Puissance Maximum GW |
| ------- | ---------------------| ---  
| C1      | 35 M&euro;          | 9000  
| C2      | 24 M&euro;          | 6000   
| C3      | 14 M&euro;          | 3000  
| C4      | 17 M&euro;          | 4000   
| C5      | 23 M&euro;          | 7000   


L'entreprise souhaite déterminer la production d’électricité par centrale ainsi que la répartition de la production en fonction des trois villes de telle manière que le coût soit minimal. 

Donner une formulation générique du problème et implémenter la en Python.


**Attention :** Il y a des variables binaires et des variables continues.

**CORRECTION :**


On peut modéliser le problème de la manière suivante.




Données : 
* On note par $C$ l'ensemble des centrales et par $V$ l'ensemble des villes.
* On note par $f_c$ le coût fixe d'utilisation de la centrale $c \in C$.
* On note $p_c^v$ le prix pour produire un GW de la centrale $c \in C$ à la ville $v \in V$.
* On note par $q_c$ la production maximum de la centrale $c \in C$.
* On note par $d^v$ la demande de la ville $v \in V$.

Variables : 
* On note par $x_c^v$ la puissance délivrée de la centrale $c \in C$ à la ville $v \in V$.
* On associe à chaque centrale $c \in C$ une variable binaire $y_c$ telle que :
  $$
  y_c = \left\{\begin{array}{ll}1 & \text{si la centrale $c$ est utilisée,}\\ 0 & \text{sinon.}\end{array} \right.
  $$

Objectif : 
*  Le but est de minimiser le coût total de production, donc la somme des coûts de chaque centrale pour l'ensemble des villes plus le coût fixe d'utilisation :  
  $$
  \min \sum_{c \in C} \sum_{v \in V} p_c^v x_c^v + \sum_{c \in C} f_c y_c
  $$

Contraintes :
* pour chaque centrale $c \in C$, la puissance délivrée à toutes les villes ne peut pas dépasser sa production maximum si elle est utilisée, et doit être zéro autrement :
  $$
  \sum_{v \in V} x_c^v \leq q_c y_c \quad \quad \text{pour toute centrale $c \in C$}
  $$
* pour chaque ville $v \in V$, la puissance délivrée par toutes les centrales doit être égale à sa demande :
  $$
  \sum_{c \in C} x_c^v = d^v \quad \quad \text{pour toute ville $v \in V$}
  $$
* les quantités ne peuvent pas être négatives.

In [ ]:
#############
#  DONNÉES  #
#############

Centrales = ["C1", "C2", "C3", "C4", "C5"]
PuissancesMax = [9000, 6000, 3000, 4000, 7000]
CoutsFixes = [35000, 24000, 14000, 17000, 23000]
Villes = ["Villetaneuse", "Épinay", "Saint-Denis"]
Demandes = [3000, 6000, 5000]

Couts = [
    [80, 100, 120],
    [20,  30,  40],
    [40,  60,  80],
    [70,  90,  40],
    [30,  30,  50]
]


In [ ]:
##################
#   Correction   #
##################


def electricite(Centrales, Villes, PuissancesMax, CoutsFixes, Demandes, Couts):
    V = range(len(Villes))
    C = range(len(Centrales))
    
    m = Model()
    m.verbose = 0

    # x[c][v] représente la puissance électrique fournie par la centrale n° c à la ville n° v
    x = [
        [m.add_var() for v in V] for c in C
    ]
    
    # y[c] = 1 si la centrale c est construite, 0 sinon
    y = [m.add_var(var_type = BINARY) for c in C]

    m.objective = minimize(xsum(CoutsFixes[c] * y[c] for c in C) + xsum(Couts[c][v] * x[c][v] for c in C for v in V))
    #@objective(m, Min, sum(Coûts[i,j] * x[i,j] for i in C, j in V) +
    #    sum(CoûtsFixes[i] * y[i] for i in C))
    
    for c in C:
        # Si la centrale n° c est construite, alors la puissance desservie par la centrale doit être 
        # inférieure ou égale à sa puissance Max. Si elle n'est pas construite, elle ne peut pas desservir
        # d'electricité.
        m += xsum(x[c][v] for v in V) <= PuissancesMax[c] * y[c]

    for v in V:
        # La puissance desservie par toutes les centrales à la ville n°j doit être égale à sa demande.
        m += xsum(x[c][v] for c in C) == Demandes[v]

    status = m.optimize()
    if status == OptimizationStatus.OPTIMAL:
        print(f"Optimum {m.objective_value}")

        for c in C:
            print(f"La centrale {Centrales[c]} est construite. Elle dessert")
            for v in V:
                if x[c][v].x >= 1e-5:
                    print(f"\t- {x[c][v].x} GW à la ville {Villes[v]}")
    else:
        print("Problème non réalisable")

electricite(Centrales, Villes, PuissancesMax, CoutsFixes, Demandes, Couts)    

## Exercice 3 : Recrutement

Les ressources humaines d'une entreprise doivent gérer l'embauche pour le mois d'avril. Chaque jour, il leur faut un certain nombre d'employés par créneau horaire, résumé dans le tableau ci-dessous.


| Créneaux | Nombre de personnes |
| -------  | --------------------|
| 6h-8h    | 48                  |
| 8h-10h   | 79                  |
| 10h-12h  | 65                  |
| 12h-14h  | 87                  |
| 14h-16h  | 64                  |
| 16h-18h  | 73                  |
| 18h-20h  | 82                  |
| 20h-22h  | 43                  |
| 22h-24h  | 52                  |
| 24h- 6h  | 15                  |

Pour cela, elle peut recruter des personnes sur différents postes de travail


| Postes | Plage horaire | Tarif journalier |
| -------| --------------| -----------------|
| Poste 1| 6h-14h        | 170&euro;        |
| Poste 2| 8h-16h        | 160&euro;        |
| Poste 3|12h-20h        | 175&euro;        |
| Poste 4|16h-24h        | 180&euro;        |
| Poste 5| 22h-6h        | 195&euro;        |

Déterminer le recrutement journalier de manière à avoir suffisamment de personnes à chaque créneau horaire et de minismiser les coûts de recrutement.



**CORRECTION :**


Le problème peut être modélisé de la manière suivante.

Données : 
* On note par $P$ l'ensemble des postes.
* On note par $H$ l'ensemble des créneaux horaires.
* On note par $t_p$ le tarif de recrutement d'une personne au poste $p \in P$.
* On note par $d^h$ le nombre de personnes devant travailler durant le créneau horaire $h \in H$.
* On note par $w_p^h$ la constante qui vaut 1 si une personne recrutée sur le poste $p \in P$ travaille au créneau horaire $h \in H$, et 0 sinon.

Variables : 
* On note par $x_p$ la variable entière correspondant au nombre de personnes recrutées sur le poste $p \in P$.

Objectif : 
* On veut minimiser le coût total de recrutement, c'est-à-dire la somme des coûts de recrutement sur chaque poste :
  $$
  \min \sum_{p \in P} t_p x_p
  $$

Contraintes : 
* Pour chaque créneau horaire $h \in H$, on veut que le nombre de personnes travaillant sur les différents postes couvrant ce créneau soit supérieur ou égal au besoin $d^h$:
  $$
  \sum_{p \in P} w_p^h x_p \ge d^h \quad \quad \text{pour tout créneau horaire $h \in H$}.
  $$
* Le nombre de personnes recrutées sur un poste ne peut pas être négatif.

In [ ]:
#############
#  DONNÉES  #
#############

# Nombre de personnes pour chaque créneau horaire
demandes = [48, 79, 65, 87, 64, 73, 82, 43, 52, 15]

# Tarifs pour chaque poste
tarifs = [170, 160, 175, 180, 195]

# horairesPostes[p,h] = 1 si le poste n°p couvre l'horaire n°h
horairesPostes = [
    [1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 0],
    [0, 0, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 0, 1]
];

In [ ]:
##################
#   Correction   #
##################


def horaires(demandes, horairesPostes, tarifs):

    m = Model()
    m.verbose = 0

    P = range(len(tarifs))
    H = range(len(demandes))
    
    # x[p] indique le nombre de personnes recrutées au poste n°p
    x = [m.add_var(var_type=INTEGER) for p in P]

    m.objective = minimize(xsum(tarifs[p] * x[p] for p in P))
    
    for h in H:
        m += xsum(horairesPostes[h][p] * x[p] for p in P) >= demandes[h]
        
    status = m.optimize()
    if status == OptimizationStatus.OPTIMAL:
        print(f"Coût du recrutement : {m.objective_value} euros")

        for p in P:
            print(f"- {round(x[p].x)} personnes recrutées sur le poste n°{p}")
    else:
        print("Pb irréalisable")
horaires(demandes, horairesPostes, tarifs)

## Exercice 4 : Sudoku


Le but de cet exercice est de résoudre cette grille de sudoku **à l'aide de la programmation linéaire en nombres entiers !**

<img src="img/grille_sudoku.png" alt="Grille de sudoku" style="width: 400px;"/>

**CORRECTION :**


Le problème peut être modélisé de la manière suivante. 

Variables : 
* Pour toute case $(i,j)$ de la grille et pour tout nombre $k \in \{1,\dots,9\}$, on considère la variable binaire $x_{i,j,k}$ qui vaut 1 si le nombre $k$ est dans cette case, et 0 sinon.

Objectif : 
* Il n'y a pas d'objectif précis puisque l'on cherche une solution, pas une meilleure solution. Pour simplifier, on cherche une solution où 1 n'est pas dans la première case :
  $$
  \min x_{1,1,1}
  $$

Contraintes : 
* On a au plus un nombre par case :
  $$
  \sum_{k \in K} x_{i,j,k} = 1 \quad \quad \text{pour toute case $(i,j)$}
  $$
* chaque nombre apparaît une fois par ligne :
  $$
  \sum_{j \in J} x_{i,j,k} = 1 \quad \quad \text{pour toute ligne $i$ et pour tout nombre $k \in K$}
  $$
* chaque nombre apparaît une fois par colonne :
  $$
  \sum_{i \in I} x_{i,j,k} = 1 \quad \quad \text{pour toute colonne $j$ et pour tout nombre $k \in K$}
  $$
* chaque nombre apparaît une fois par carré 3 par 3 :
  $$
  \sum_{(i,j) \in C} x_{i,j,k} = 1 \quad \quad \text{pour tout carré $C$ et pour tout nombre $k \in K$}
  $$
* Les variables doivent prendre la valeur de la case pour les cases non vides.

In [ ]:
#############
#  DONNÉES  #
#############

grille = [
  [5, 3, 0, 0, 7, 0, 0, 0, 0],
  [6, 0, 0, 1, 9, 5, 0, 0, 0],
  [0, 9, 8, 0, 0, 0, 0, 6, 0],
  [8, 0, 0, 0, 6, 0, 0, 0, 3],
  [4, 0, 0, 8, 0, 3, 0, 0, 1],
  [7, 0, 0, 0, 2, 0, 0, 0, 6],
  [0, 6, 0, 0, 0, 0, 2, 8, 0],
  [0, 0, 0, 4, 1, 9, 0, 0, 5],
  [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

In [ ]:
##################
#   Correction   #
##################


def sudoku(grille):

    m = Model()
    m.verbose = 0
    
    N = range(len(grille))
    
    # x[i,j,k] = 1 si la case (i,j) contient la valeur k, 0 sinon
    x = [[[m.add_var(var_type=BINARY) for k in N] for j in N] for i in N]


    m.objective = minimize(x[0][0][0])

    
    # Un nombre par case
    for i in N:
        for j in N:
            m += xsum(x[i][j][k] for k in N) == 1
    
    # Un nombre apparaît une seule fois par ligne
    for i in N:
        for k in N:
            m += xsum(x[i][j][k] for j in N) == 1
    
    # Un nombre apparaît une seule fois par colonne
    for j in N:
        for k in N:
            m += xsum(x[i][j][k] for i in N) == 1
    
    # Un nombre apparaît une seule fois par carré
    for c1  in range(3):
        for c2 in range(3):
            for k in N:
                m += xsum(x[3 * c1 + i][3 * c2 + j][k] for i in range(3) for j in range(3)) == 1


    #On fixe les valeurs de la grille
    for i in N:
        for j in N:
            if grille[i][j] != 0:
                m += x[i][j][grille[i][j] - 1] == 1
    
    status = m.optimize()
    if status == OptimizationStatus.OPTIMAL:
        print("La grille admet une solution\n")

        # On récupère les valeurs des variables
        resolue = [[0 for j in N] for i in N]      
        
        for i in N:
            for j in N:
                for k in N:
                    if round(x[i][j][k].x) == 1:
                        resolue[i][j] = k + 1
        for i in N:
            print(" ".join(map(str,resolue[i])))
    else:
        print("La grille n'admet pas de solution")

sudoku(grille)